In [1]:
import pandas as pd
import numpy as np

In [3]:
# 1. BACA DATA NASA POWER
nasa_file = "POWER_Point_Daily_20200101_20261115_001d29S_113d41E_LST.csv"

with open(nasa_file, "r") as f:
    for i, line in enumerate(f):
        if "-END HEADER-" in line:
            skip_rows = i + 1
            break

df_cuaca = pd.read_csv(nasa_file, skiprows=skip_rows)
df_cuaca.replace(-999, np.nan, inplace=True)
df_cuaca.dropna(inplace=True)

# File ini punya kolom YEAR dan DOY (hari ke-n dalam setahun), bukan MO dan DY
df_cuaca["date"] = pd.to_datetime(
    df_cuaca["YEAR"].astype(str) + df_cuaca["DOY"].astype(str).str.zfill(3),
    format="%Y%j"
).dt.strftime("%Y-%m-%d")

df_cuaca = df_cuaca[["date", "T2M", "RH2M", "PRECTOTCORR", "GWETTOP"]]

In [5]:
# 2. BACA & FILTER DATA HOTSPOT (Sesuai kolom daily_province.csv)
df_hotspot = pd.read_csv("daily_province.csv", sep=";")

# Ganti sesuai provinsi yang titik koordinatnya kamu ambil di NASA POWER
nama_provinsi = "Central Kalimantan"

df_hotspot_prov = df_hotspot[df_hotspot["PROVINCE"] == nama_provinsi].copy()

In [6]:
# Jumlahkan hotspot dari satelit SNPP dan NOAA20
df_hotspot_prov["TOTAL_HOTSPOTS"] = (
    df_hotspot_prov["HOTSPOTS_SNPP"].fillna(0) +
    df_hotspot_prov["HOTSPOTS_NOAA20"].fillna(0)
)

# Parse tanggal format DD/MM/YYYY
df_hotspot_prov["date"] = pd.to_datetime(
    df_hotspot_prov["DATE"], dayfirst=True
).dt.strftime("%Y-%m-%d")

df_hotspot_prov = df_hotspot_prov[["date", "TOTAL_HOTSPOTS"]]

In [7]:
# 3. GABUNGKAN (LEFT JOIN) BERDASARKAN TANGGAL
df_final = pd.merge(df_cuaca, df_hotspot_prov, on="date", how="left")
df_final["TOTAL_HOTSPOTS"] = df_final["TOTAL_HOTSPOTS"].fillna(0).astype(int)

In [8]:
# 4. PELABELAN MULTI-KELAS (Aman, Waspada, Bahaya)
def tentukan_risiko(jumlah):
    if jumlah == 0:
        return "Aman"
    elif jumlah <= 10:
        return "Waspada"
    else:
        return "Bahaya"

df_final["Status_Risiko"] = df_final["TOTAL_HOTSPOTS"].apply(tentukan_risiko)

In [9]:
df_final.rename(columns={
    "T2M": "Suhu_Udara",
    "RH2M": "Kelembapan_Udara",
    "PRECTOTCORR": "Curah_Hujan",
    "GWETTOP": "Kelembapan_Tanah"
}, inplace=True)

print(df_final.head(10))
print("\nDistribusi Kelas Risiko:")
print(df_final["Status_Risiko"].value_counts())

         date  Suhu_Udara  Kelembapan_Udara  Curah_Hujan  Kelembapan_Tanah  \
0  2020-01-01       26.87             84.99         8.98              0.76   
1  2020-01-02       26.78             88.41        19.92              0.81   
2  2020-01-03       25.81             92.55        11.87              0.78   
3  2020-01-04       28.29             81.36        63.20              0.89   
4  2020-01-05       27.18             85.92        78.23              0.89   
5  2020-01-06       26.49             89.23         7.77              0.80   
6  2020-01-07       26.81             83.55         2.64              0.77   
7  2020-01-08       27.22             83.20         1.11              0.76   
8  2020-01-09       27.33             86.08         8.40              0.78   
9  2020-01-10       25.87             93.10        12.30              0.81   

   TOTAL_HOTSPOTS Status_Risiko  
0               0          Aman  
1               0          Aman  
2               0          Aman  
3    

In [10]:
# Simpan menjadi dataset siap latih
df_final.to_csv("dataset_karhutla_siap_latih.csv", index=False)